# Spark

## Installing PySpark

In [ ]:
!pip install pyspark

## Imports

In [ ]:
import typing

import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas
import pyspark
import pyspark.ml
import pyspark.sql.functions as F
import seaborn as sns

## Loading the Data

In this lab, we'll use real estate transaction data for all of France, between 2014 and 2022.

This data comes from the [French open data](https://www.data.gouv.fr/fr/datasets/demandes-de-valeurs-foncieres-geolocalisees/) website. We'll use a reworked version that groups transactions, from the [NyxAether/DVF](https://github.com/NyxAether/DVF) repository on GitHub.

In [ ]:
!wget -qO- https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh | bash -s dvf

In Spark SQL, you always start by defining a session, named `spark` by convention. This session specifies how to access the cluster, a few options, etc. Here, we'll use it in its simplest form.

In [ ]:
spark = pyspark.sql.SparkSession.builder.appName("DVF").getOrCreate()

*You can load the data with the [`pyspark.sql.DataFrameReader.parquet`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameReader.parquet.html) function. You can call this function with the syntax `spark.read.parquet(...)`, where `spark` is the Spark session we just created and `...` your arguments.*

In [ ]:
# Your code here

*Once the data is available in a Spark dataframe, create a `date` column from the `annee_mutation`, `mois_mutation` & `jour_mutation` columns with the [`pyspark.sql.functions.make_date`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.make_date.html) function, then drop these columns to save memory.*

In [ ]:
# Your code here

Finally, display the column types of the dataframe.

In [ ]:
# Your code here

### Solution

In [ ]:
df = spark.read.parquet("dvf/dvf-linearized-2014-2022.parquet")

In [ ]:
df = df.withColumn(
  "date", F.make_date("annee_mutation", "mois_mutation", "jour_mutation")
).drop("annee_mutation", "mois_mutation", "jour_mutation")
df.show(10)

In [ ]:
df.printSchema()

## Number of Transactions per Year

To start, we'll count the number of transactions (rows of the dataset) per year.

*Use the following functions to provide the data to the chart creation function:*
- *The [`pyspark.sql.functions.year`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.year.html) function to use the `date` column.*
- *The [`pyspark.sql.DataFrame.groupBy`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.groupBy.html) function to group the data.*
- *The [`pyspark.sql.DataFrame.toPandas`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.toPandas.html) function to return a data structure seaborn can easily use.*

The chart creation function expects the data in a pandas dataframe organized like this:

```
   year    count
0  2014  1032967
1  2015  1128516
2  2016  1210587
3  2017  1369101
4  2018  1351856
5  2019  1431358
6  2020  1268193
7  2021  1537648
8  2022   508837
```

In [ ]:
def get_data_transactions_by_year() -> pandas.DataFrame:
  pass  # Your code here


def transactions_by_year() -> None:
  data = get_data_transactions_by_year()

  fig, ax = plt.subplots()
  sns.barplot(data, x="year", y="count", ax=ax)
  sns.despine(fig)
  ax.set_title("Transaction count per year")
  ax.set_xlabel("Year")
  ax.set_ylabel("Number of transactions")
  ax.get_yaxis().set_major_formatter(mpl.ticker.EngFormatter(places=1))
  fig.show()


# transactions_by_year()

### Solution

In [ ]:
def get_data_transactions_by_year() -> pandas.DataFrame:
  return df.groupby(F.year("date").alias("year")).count().orderBy("year").toPandas()


def transactions_by_year() -> None:
  data = get_data_transactions_by_year()

  fig, ax = plt.subplots()
  sns.barplot(data, x="year", y="count", ax=ax)
  sns.despine(fig)
  ax.set_title("Transaction count per year")
  ax.set_xlabel("Year")
  ax.set_ylabel("Number of transactions")
  ax.get_yaxis().set_major_formatter(mpl.ticker.EngFormatter(places=1))
  fig.show()


transactions_by_year()

## Total Property Value by Department

In this exercise, we'll group our data by the values of a column (here the department of the transaction) and display the total property value exchanged per group.

- *Group the data by department with the [`pyspark.sql.DataFrame.groupBy`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.groupBy.html) function.*
- *Sort the result in decreasing order with the [`pyspark.sql.DataFrame.orderBy`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.orderBy.html) function.*
- *Convert the result into a pandas series (with the departments as index).*

The expected pandas series looks like this:

```
code_departement
75    1.821853e+11
92    1.327795e+11
6     8.143718e+10
33    8.015320e+10
69    7.632324e+10
          ...     
55    1.998241e+09
15    1.907638e+09
52    1.753427e+09
23    1.248303e+09
48    8.985644e+08
Name: total, Length: 97, dtype: float64
```

In [ ]:
def get_data_value_by_department() -> pandas.Series:
  pass  # Your code here


def value_by_department() -> None:
  data = get_data_value_by_department()

  fig, ax = plt.subplots(figsize=(30, 10))
  sns.barplot(x=data.index, y=data, order=data.index, ax=ax)
  sns.despine(ax=ax)
  ax.set_title("Total transaction amount per department")
  ax.set_xlabel("Department")
  ax.set_ylabel("Total transaction amount")
  ax.get_yaxis().set_major_formatter(mpl.ticker.EngFormatter(unit="€", places=1))
  fig.show()


# value_by_department()

### Solution

In [ ]:
def get_data_value_by_department() -> pandas.Series:
  return (
    df.groupby("code_departement")
    .agg(F.sum("valeur_fonciere").alias("total"))
    .orderBy("total", ascending=False)
    .toPandas()
    .set_index("code_departement")
    .iloc[:, 0]
  )


def value_by_department() -> None:
  data = get_data_value_by_department()

  fig, ax = plt.subplots(figsize=(30, 10))
  sns.barplot(x=data.index, y=data, order=data.index, ax=ax)
  sns.despine(ax=ax)
  ax.set_title("Total transaction amount per department")
  ax.set_xlabel("Department")
  ax.set_ylabel("Total transaction amount")
  ax.get_yaxis().set_major_formatter(mpl.ticker.EngFormatter(unit="€", places=1))
  fig.show()


value_by_department()

## Total Property Value by Region and by Year

In this part, we'll study how to break down a categorical variable by another variable: we'll produce a chart of the total property value exchanged by region, and this value will itself be broken down by year.

To do so, let's start by fetching a dataset that will let us group departments into regions:

In [ ]:
!wget https://github.com/shuuchuu/datasets/raw/refs/heads/main/france-departements/departements-region.csv -O regions.csv

The following function returns a Spark dataframe that makes it easy to join the region to our working dataframe.

In [ ]:
def get_regions_df() -> pyspark.sql.DataFrame:
  return (
    spark.read.csv("regions.csv", inferSchema=True, header=True)
    .drop("dep_name")
    .withColumn("num_dep", F.regexp_replace("num_dep", "^0+", ""))
  )


get_regions_df().show(10)

- *Add a column containing the region name to your working dataframe, with the [`pyspark.sql.DataFrame.join`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.join.html) function.*
- *Return a pandas dataframe containing the sum of the property value exchanged per region and per year, with the [`pyspark.sql.DataFrame.groupBy`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.groupBy.html) function.*
- *Also return a pandas series: the region names sorted from the region with the largest sum of property value exchanged to the smallest.*

The dataframe to return looks like this:

```
                 region_name  year         total
0              Île-de-France  2021  9.168523e+10
1    Bourgogne-Franche-Comté  2021  8.240557e+09
2                 La Réunion  2021  1.681132e+09
3                     Guyane  2021  5.290445e+08
4                   Bretagne  2021  1.613739e+10
..                       ...   ...           ...
148          Hauts-de-France  2014  1.532841e+10
149            Île-de-France  2014  5.324466e+10
150                Normandie  2022  2.027011e+09
151  Bourgogne-Franche-Comté  2014  4.475983e+09
152                 Bretagne  2014  6.876292e+09
```

The series to return looks like this:

```
0                  Île-de-France
1           Auvergne-Rhône-Alpes
2     Provence-Alpes-Côte d'Azur
3             Nouvelle-Aquitaine
4                      Occitanie
5                Hauts-de-France
6               Pays de la Loire
7                       Bretagne
8                      Normandie
9            Centre-Val de Loire
10       Bourgogne-Franche-Comté
11                     Grand Est
12                    La Réunion
13                         Corse
14                    Guadeloupe
15                    Martinique
16                        Guyane
Name: region_name, dtype: object
```

In [ ]:
def get_data_value_by_region_by_year() -> tuple[pandas.DataFrame, pandas.Series]:
  pass  # Your code here


def value_by_region_by_year() -> None:
  data, sorted_index = get_data_value_by_region_by_year()

  fig, ax = plt.subplots(figsize=(20, 10))
  sns.barplot(data, x="region_name", y="total", hue="year", order=sorted_index, ax=ax)
  sns.despine(fig)
  ax.set_title("Total transaction amount by region and by year")
  ax.set_xlabel("Region")
  plt.xticks(rotation=60)
  ax.set_ylabel("Total transaction amount")
  ax.get_yaxis().set_major_formatter(mpl.ticker.EngFormatter(unit="€", places=1))
  fig.show()


# value_by_region_by_year()

### Solution

In [ ]:
def get_data_value_by_region_by_year() -> tuple[pandas.DataFrame, pandas.Series]:
  regions_df = get_regions_df()
  joined = df.join(regions_df, df.code_departement == regions_df.num_dep).drop(
    "num_dep"
  )
  data = (
    joined.groupBy(["region_name", F.year("date").alias("year")])
    .agg(F.sum("valeur_fonciere").alias("total"))
    .toPandas()
  )
  sorted_index = (
    joined.groupBy("region_name")
    .agg(F.sum("valeur_fonciere").alias("total"))
    .orderBy("total", ascending=False)
    .select("region_name")
    .toPandas()
    .iloc[:, 0]
  )
  return data, sorted_index


def value_by_region_by_year() -> None:
  data, sorted_index = get_data_value_by_region_by_year()

  fig, ax = plt.subplots(figsize=(20, 10))
  sns.barplot(data, x="region_name", y="total", hue="year", order=sorted_index, ax=ax)
  sns.despine(fig)
  ax.set_title("Total transaction amount by region and by year")
  ax.set_xlabel("Region")
  plt.xticks(rotation=60)
  ax.set_ylabel("Total transaction amount")
  ax.get_yaxis().set_major_formatter(mpl.ticker.EngFormatter(unit="€", places=1))
  fig.show()


value_by_region_by_year()

## House and Apartment Prices

In this exercise, we'll compute the price per square meter of apartments and houses separately, and display how these prices evolve over the period considered.

- *For apartments, filter the working dataframe to keep only the rows where:*
    - *`nombre_appartements`, `valeur_fonciere` & `surface_reelle_bati_appartements` are strictly positive.*
    - *`nombre_maisons` is `0`.*
- *Likewise, for houses, filter the working dataframe to keep only the rows where:*
    - *`nombre_maisons`, `valeur_fonciere` & `surface_reelle_bati_maisons` are strictly positive.*
    - *`nombre_appartements` is `0`.*
- *Use this filtered data to compute the price per square meter for each month available in the dataset, for houses and apartments separately. You can use the following functions:*
    - The [`pyspark.sql.functions.date_format`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.date_format.html) function to get a column containing the date in the `yyyy-MM` form.
    - The [`pyspark.sql.DataFrame.groupBy`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.groupBy.html) function
- *The chart function used, [`seaborn.lineplot`](https://seaborn.pydata.org/generated/seaborn.lineplot.html), expects data in long form. Use the [`pyspark.sql.DataFrame.melt`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.melt.html) function to provide it.*

The expected pandas dataframe looks like this:

```
       month       type        price
0    2014-01  apartment  3005.224170
1    2014-01      house  1588.628790
2    2014-02  apartment  2878.714589
3    2014-02      house  1622.620962
4    2014-03  apartment  2911.870507
..       ...        ...          ...
199  2022-04      house  2190.928353
200  2022-05  apartment  4164.208224
201  2022-05      house  2150.579262
202  2022-06  apartment  4530.162208
203  2022-06      house  2242.808145
```

In [ ]:
def get_data_houses_apartments_prices() -> pandas.DataFrame:
  pass  # Your code here


def houses_apartments_prices() -> None:
  data = get_data_houses_apartments_prices()

  fig, ax = plt.subplots()
  sns.lineplot(data, x="month", y="price", hue="type", legend=True)
  plt.xticks(rotation=45)
  ax.set_xticks(ax.get_xticks()[::12])
  ax.set_xlabel("Date")
  ax.set_ylabel("Average value")
  ax.set_title("Price per square meter")
  ax.set_ylim((0, data.select_dtypes("number").max().max()))
  ax.get_yaxis().set_major_formatter(mpl.ticker.EngFormatter(unit="€/m²", places=1))
  fig.show()


# houses_apartments_prices()

### Solution

In [ ]:
def get_data_houses_apartments_prices() -> pandas.DataFrame:
  mask_apartments = (
    (df.nombre_appartements > 0)
    & (df.nombre_maisons == 0)
    & (df.valeur_fonciere > 0)
    & (df.surface_reelle_bati_appartements > 0)
  )
  mask_houses = (
    (df.nombre_appartements == 0)
    & (df.nombre_maisons > 0)
    & (df.valeur_fonciere > 0)
    & (df.surface_reelle_bati_maisons > 0)
  )

  def get_prices(
    mask: typing.Any, area_column: str, column_name: str
  ) -> pyspark.sql.DataFrame:
    return (
      df.filter(mask)
      .groupby(F.date_format("date", "yyyy-MM").alias("month"))
      .agg(F.sum("valeur_fonciere"), F.sum(area_column))
      .withColumn(
        column_name,
        F.col("sum(valeur_fonciere)") / F.col(f"sum({area_column})"),
      )
      .drop("sum(valeur_fonciere)", f"sum({area_column})")
    )

  prices_apartments = get_prices(
    mask_apartments, "surface_reelle_bati_appartements", "apartment"
  )
  prices_houses = get_prices(mask_houses, "surface_reelle_bati_maisons", "house")

  return (
    prices_apartments.join(prices_houses, "month")
    .orderBy("month")
    .melt(
      ids="month",
      values=["apartment", "house"],
      variableColumnName="type",
      valueColumnName="price",
    )
    .toPandas()
  )


def houses_apartments_prices() -> None:
  data = get_data_houses_apartments_prices()

  fig, ax = plt.subplots()
  sns.lineplot(data, x="month", y="price", hue="type", legend=True)
  plt.xticks(rotation=45)
  ax.set_xticks(ax.get_xticks()[::12])
  ax.set_xlabel("Date")
  ax.set_ylabel("Average value")
  ax.set_title("Price per square meter")
  ax.set_ylim((0, data.select_dtypes("number").max().max()))
  ax.get_yaxis().set_major_formatter(mpl.ticker.EngFormatter(unit="€/m²", places=1))
  fig.show()


houses_apartments_prices()

## Correlations Between Variables

To finish, we'll see how to compute the linear correlations with the property value, and more generally between variables, in Spark. The process is much more complex than in pandas, but it will let us see how MLlib works.

In [ ]:
def get_data_corrs() -> pandas.DataFrame:
  vector_col = "corr_features"

  corr_columns = [
    f.name
    for f in df.schema.fields
    if isinstance(f.dataType, pyspark.sql.types.NumericType)
  ]

  assembler = pyspark.ml.feature.VectorAssembler(
    inputCols=corr_columns, outputCol=vector_col
  )
  df_vector = assembler.transform(df).select(vector_col)
  matrix = pyspark.ml.stat.Correlation.corr(df_vector, vector_col)
  values = matrix.collect()[0][f"pearson({vector_col})"].values

  result = pandas.DataFrame(
    values.reshape(-1, len(corr_columns)), columns=corr_columns, index=corr_columns
  )
  index = result.valeur_fonciere.sort_values(ascending=False, key=abs).index
  return result.loc[index, index]


def corrs() -> None:
  data = get_data_corrs()
  print(data)
  fig, ax = plt.subplots(figsize=(15, 10))
  sns.heatmap(data, vmin=-1, vmax=1, cmap="vlag", ax=ax)
  ax.set_title("Correlations between variables sorted by correlation to the price")
  fig.show()


corrs()